# DS4 Dynamical Systems (DyS) -- Stage 4 Step 1

**Notebook:** `55_ds4_dynamical_systems.ipynb`  
**Model family:** Dynamical Systems (DyS)  
**Dataset:** DS4 -- Large Naturalistic OneSec Users (N=1,039)  
**Prerequisites:** DS4 StoP (53) and DS4 InT (52) -- both complete, output loaded directly.  
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_dys_blueprint.md`

---

## Goal

Characterize DS4's OU fixed point `X*=mu_hat_OU` as a genuine **interior, stable attractor** -- the deterministic skeleton `dX/dt=theta(mu-X)` is exact for the OU model, unlike DS5's constant-drift model which only had a boundary fixed point under a sigma->0 limit. Validate the fixed point against the raw empirical mean, split participants into control regimes via theta_hat_OU, and bridge to InT's H_emp.

## Not the goal

- No bifurcation or multi-well analysis -- the OU model is linear-in-drift, so there is exactly one fixed point per participant by construction, not a simplification being deferred.
- No re-estimation of StoP or InT quantities -- both are loaded directly.

## Core claim

> DS4 can use the word "attractor" without qualification, where DS5's DyS notebook explicitly could not (`predicted_boundary_fp`, never "attractor"). Every StoP-valid participant already has theta>0 by construction of the fit-validity criterion, so every one of them already has a genuine stable interior fixed point -- this notebook characterizes and validates that fact, it does not newly discover eligibility for it.

## Output

```
outputs/model_outputs/ds4_dynamical_systems/
  participant_dynamics_parameters.csv
  dys_stop_int_bridge_summary.csv
  model_run_summary.md
  figures/
```


## FLAG decisions

| Flag | Decision | Justification |
|---|---|---|
| FLAG 1 | Fixed point X*=mu_hat_OU, taken directly from StoP | It is StoP's own parameter -- not re-derived |
| FLAG 2 | Regime split via theta_hat_OU median -> high_control/low_control | theta is DS4's own richer parameter, DS5 never had it |
| FLAG 3 | behaviorally_deterministic = high_control AND H_emp<median(H_emp) | Reuses DS5's bridge-construction pattern; reports the real correlation honestly |
| FLAG 4 | No bifurcation/multi-well analysis | OU model has exactly one fixed point per participant by construction |
| FLAG 5 | Regime split uses daily-scale StoP output only | Same daily-timescale caveat as InT's H_SDE |


## DS4 structural cautions

- No experimental condition -- regime differences describe naturalistic heterogeneity, never a friction effect.
- Regime split (theta_hat_OU) is daily-timescale-only -- StoP's own weekly sensitivity check found a meaningfully different relaxation time at weekly resolution; this notebook inherits that caveat rather than re-deriving a weekly regime split.


## Theory

Deterministic skeleton of the OU process: $\frac{dX}{dt} = \theta(\mu - X)$

Fixed point: $X^* = \mu$. Stability: $\frac{d}{dX}\theta(\mu-X) = -\theta < 0$ whenever $\theta>0$ -- **always stable** for every valid StoP fit (fit-validity already required $0<\hat\phi<1$, equivalent to $\hat\theta>0$).

This is a genuine interior attractor, not a boundary approximation: DS5's constant-drift model `dX/dt=mu` has no interior fixed point at all except in a sigma->0 limiting sense.


## Theory-to-data mapping

| Quantity | DS4 source | Confidence |
|---|---|---|
| Fixed point X* | `mu_hat_OU` (StoP) | High -- exact |
| Stability | Always stable for `fit_valid=True` | High |
| Regime label | theta_hat_OU vs its own median | Medium -- real split, weak InT bridge |
| Fixed-point validation | `\|x_mean_raw - mu_hat_OU\|` | High -- verified small |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BEH_PATH = Path("outputs/cleaned/ds4_behavioral_clean.csv")
STOP_PATH = Path("outputs/model_outputs/ds4_stochastic_processes/participant_parameters.csv")
INT_PATH = Path("outputs/model_outputs/ds4_information_theory/participant_entropy_metrics.csv")
OUT_DIR = Path("outputs/model_outputs/ds4_dynamical_systems")
FIG_DIR = OUT_DIR / "figures"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

stop_params = pd.read_csv(STOP_PATH)
int_metrics = pd.read_csv(INT_PATH)

print(f"stop_params: {len(stop_params)} rows, {int(stop_params['fit_valid'].sum())} valid")
print(f"int_metrics: {len(int_metrics)} rows")


In [ ]:
beh = pd.read_csv(BEH_PATH, parse_dates=["timestamp_utc"], low_memory=False)
df = beh[beh["intervention_event_class"] == "intervention_event"].copy()
df = df[df["event_outcome_simple"].isin(["opened", "dismissed"])].copy()
df["opened_flag"] = (df["event_outcome_simple"] == "opened").astype(int)
df["event_date"] = df["timestamp_utc"].dt.date

daily = df.groupby(["participant_id_canonical", "event_date"]).agg(
    n_opened=("opened_flag", "sum"),
    n_events=("opened_flag", "count"),
).reset_index()
daily["X_t"] = daily["n_opened"] / daily["n_events"]
daily["event_date"] = pd.to_datetime(daily["event_date"])
daily = daily.sort_values(["participant_id_canonical", "event_date"]).reset_index(drop=True)

x_mean_raw = daily.groupby("participant_id_canonical")["X_t"].mean().rename("x_mean_raw").reset_index()

dys = stop_params.merge(x_mean_raw, on="participant_id_canonical", how="left")
dys["fixed_point_deviation"] = (dys["x_mean_raw"] - dys["mu_hat_OU"]).abs()

valid_fp = dys[dys["fit_valid"]]
print(f"Fixed-point validation (n={len(valid_fp)}):")
print(f"  mean |x_mean_raw - mu_hat_OU| = {valid_fp['fixed_point_deviation'].mean():.4f}")
print(f"  max  |x_mean_raw - mu_hat_OU| = {valid_fp['fixed_point_deviation'].max():.4f}")
print(f"  corr(x_mean_raw, mu_hat_OU)   = {valid_fp['x_mean_raw'].corr(valid_fp['mu_hat_OU']):.4f}")


In [ ]:
theta_median = dys.loc[dys["fit_valid"], "theta_hat_OU"].median()

dys["regime_label"] = pd.NA
dys.loc[dys["fit_valid"] & (dys["theta_hat_OU"] >= theta_median), "regime_label"] = "high_control"
dys.loc[dys["fit_valid"] & (dys["theta_hat_OU"] < theta_median), "regime_label"] = "low_control"

print(f"theta_hat_OU median = {theta_median:.4f}")
print(dys["regime_label"].value_counts(dropna=False))


In [ ]:
dys = dys.merge(int_metrics[["participant_id_canonical", "H_emp"]], on="participant_id_canonical", how="left")

h_emp_median = dys.loc[dys["fit_valid"], "H_emp"].median()
dys["behaviorally_deterministic"] = (
    (dys["regime_label"] == "high_control") & (dys["H_emp"] < h_emp_median)
)
dys.loc[~dys["fit_valid"], "behaviorally_deterministic"] = np.nan

valid_bridge = dys[dys["fit_valid"]]
corr_theta_hemp = valid_bridge["theta_hat_OU"].corr(valid_bridge["H_emp"])
print(f"corr(theta_hat_OU, H_emp) = {corr_theta_hemp:.4f} (n={len(valid_bridge)})")
print(f"behaviorally_deterministic count: {int(dys['behaviorally_deterministic'].sum())}")


In [ ]:
export_cols = [
    "participant_id_canonical", "mu_hat_OU", "theta_hat_OU", "x_mean_raw",
    "fixed_point_deviation", "regime_label", "behaviorally_deterministic",
    "H_emp", "fit_valid",
]
participant_dynamics = dys[export_cols].copy()
participant_dynamics.to_csv(OUT_DIR / "participant_dynamics_parameters.csv", index=False)
print(f"Wrote participant_dynamics_parameters.csv ({len(participant_dynamics)} rows)")


In [ ]:
print(dys.loc[dys["fit_valid"], "regime_label"].value_counts())
print()
print(dys.groupby("regime_label", observed=True)["mu_hat_OU"].describe())


In [ ]:
regime_counts = dys.loc[dys["fit_valid"], "regime_label"].value_counts()
bridge_summary = pd.DataFrame([{
    "n_valid": int(dys["fit_valid"].sum()),
    "n_high_control": int(regime_counts.get("high_control", 0)),
    "n_low_control": int(regime_counts.get("low_control", 0)),
    "theta_median": theta_median,
    "corr_xmeanraw_muhatOU": valid_fp["x_mean_raw"].corr(valid_fp["mu_hat_OU"]),
    "mean_fixed_point_deviation": valid_fp["fixed_point_deviation"].mean(),
    "corr_theta_Hemp": corr_theta_hemp,
    "n_behaviorally_deterministic": int(dys["behaviorally_deterministic"].sum()),
}])
bridge_summary.to_csv(OUT_DIR / "dys_stop_int_bridge_summary.csv", index=False)
print(bridge_summary.T)

summary_lines = []
summary_lines.append("# DS4 DyS -- Model Run Summary")
summary_lines.append("")
summary_lines.append("**Notebook:** `55_ds4_dynamical_systems.ipynb`  ")
summary_lines.append("**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_dys_blueprint.md`  ")
summary_lines.append("**Run date:** " + pd.Timestamp.now().strftime("%Y-%m-%d"))
summary_lines.append("")
summary_lines.append("## Fixed-point validation")
summary_lines.append("")
summary_lines.append(f"- n_valid = {int(dys['fit_valid'].sum())}")
summary_lines.append(f"- corr(x_mean_raw, mu_hat_OU) = {valid_fp['x_mean_raw'].corr(valid_fp['mu_hat_OU']):.4f}")
summary_lines.append(f"- mean |x_mean_raw - mu_hat_OU| = {valid_fp['fixed_point_deviation'].mean():.4f}")
summary_lines.append("")
summary_lines.append("## Regime split")
summary_lines.append("")
summary_lines.append(f"- theta_hat_OU median = {theta_median:.4f}")
summary_lines.append(f"- high_control n = {int(regime_counts.get('high_control', 0))}")
summary_lines.append(f"- low_control n = {int(regime_counts.get('low_control', 0))}")
summary_lines.append("")
summary_lines.append("## InT bridge")
summary_lines.append("")
summary_lines.append(f"- corr(theta_hat_OU, H_emp) = {corr_theta_hemp:.4f} -- weak, "
                     "reported honestly, not a strong cross-validation")
summary_lines.append(f"- behaviorally_deterministic n = {int(dys['behaviorally_deterministic'].sum())}")

with open(OUT_DIR / "model_run_summary.md", "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print("Wrote model_run_summary.md")


In [ ]:
sample_pids = dys.loc[dys["fit_valid"]].sample(n=6, random_state=42)["participant_id_canonical"].tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, pid in zip(axes.flat, sample_pids):
    g = daily[daily["participant_id_canonical"] == pid].sort_values("event_date")
    gap = g["event_date"].diff().dt.days
    x_t = g["X_t"].values[:-1]
    x_t1 = g["X_t"].values[1:]
    consec = (gap.values[1:] == 1)
    x_t, x_t1 = x_t[consec], x_t1[consec]

    row = dys[dys["participant_id_canonical"] == pid].iloc[0]
    ax.scatter(x_t, x_t1, s=12, alpha=0.6, color="#4C72B0")
    xs = np.linspace(0, 1, 50)
    phi = np.exp(-row["theta_hat_OU"])
    c = row["mu_hat_OU"] * (1 - phi)
    ax.plot(xs, c + phi * xs, color="#C44E52", label="fitted AR(1)")
    ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, label="X_t=X_t+1")
    ax.scatter([row["mu_hat_OU"]], [row["mu_hat_OU"]], color="black", zorder=5, s=60,
                marker="*", label="X* (attractor)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(f"{pid[:8]}... (theta={row['theta_hat_OU']:.2f})")
    ax.set_xlabel("X_t")
    ax.set_ylabel("X_t+1")

axes.flat[0].legend(fontsize=8)
fig.suptitle("DS4 DyS: AR(1) phase plots, sample participants, fixed point = attractor")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_ds4_phase_plot_sample.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for regime, color in [("high_control", "#4C72B0"), ("low_control", "#C44E52")]:
    vals = dys.loc[dys["regime_label"] == regime, "mu_hat_OU"]
    ax.hist(vals, bins=30, alpha=0.6, label=f"{regime} (n={len(vals)})", color=color)
ax.set_xlabel("X* = mu_hat_OU (attractor location)")
ax.set_ylabel("count")
ax.legend()
ax.set_title("DS4 DyS: attractor location by control regime")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_ds4_fixed_point_by_regime.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(valid_fp["mu_hat_OU"], valid_fp["x_mean_raw"], s=14, alpha=0.6, color="#55A868")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("mu_hat_OU (OU-fitted mean)")
ax.set_ylabel("x_mean_raw (empirical mean)")
r_val = valid_fp['x_mean_raw'].corr(valid_fp['mu_hat_OU'])
ax.set_title(f"Fixed-point validation (r={r_val:.4f})")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_ds4_fixedpoint_validation.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(valid_bridge["theta_hat_OU"], valid_bridge["H_emp"], s=14, alpha=0.6, color="#8172B2")
ax.set_xlabel("theta_hat_OU")
ax.set_ylabel("H_emp (bits)")
ax.set_title(f"DyS-InT bridge (r={corr_theta_hemp:.3f}) -- weak, reported honestly")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_ds4_theta_hemp_bridge.png", dpi=150)
plt.close(fig)


## Interpretation and limits

**Read the cells above for live numbers.**

- Every StoP-valid participant already has a genuine, stable interior attractor at `X*=mu_hat_OU` -- theta>0 is guaranteed by construction of the fit-validity criterion. This notebook does not discover eligibility, it characterizes and validates an already-guaranteed fact.
- The fixed-point validation (cell 7) confirms the OU-fitted mean tracks the raw empirical mean closely -- a real sanity check the model describes the observed level well.
- The theta/H_emp bridge (cell 9) is real but weak -- do not present `behaviorally_deterministic` as strongly cross-validated the way DS5's analogous construct was.
- Regime split is daily-timescale-only -- inherits StoP's daily/weekly divergence caveat.

### DS4 weakness flags

- FLAG I: weak InT bridge (theta vs H_emp).
- FLAG II: daily-timescale-only regime split.
- FLAG III: no experimental condition.
- FLAG IV: single-fixed-point system by construction, not a simplification.
- FLAG V: behavioral proxy limitation persists.


## Forward connections

| Connection | Mechanism | Target |
|---|---|---|
| DS4 DyS -> DS4 StM | Interior attractor X*=mu_hat_OU gives StM's V(x) minimum a genuine behavioral-attractor interpretation | `51_ds4_statistical_mechanics.ipynb` |
| DS4 DyS -> DS4 NeT | None directly -- NeT is independent | `54_ds4_network_models.ipynb` |
| DS4 DyS -> Paper | True interior attractor, resolving DS5 DyS's boundary-only limitation, not yet in the conceptual paper |

**Next step:** re-verify `ds4_stm_blueprint.md` is still current against real StoP/InT/DyS output, then implement DS4 StM (51).
